# DATA 4381 - CAPSTONE - FER-2013

## Neural Network Baseline - Augmentation Comparison

### Overview

This notebook will evaluate our neural network build from scratch using NumPy. This notebook will focus on solving our primary issue of data imbalance across the FER-2013 dataset. To do so we implemented several forms of data augmentation and we will compare the results here.

### Classification Task

Our model predicts three facial expressions (compared to four in our original baseline):

- Angry (0)
- Happy (1)
- Sad (2)
  NOTE: Neutral was dropped (in similar guidance to Microsoft research team)

### Neural Network Architecture

This network uses a fully connected feedforward:

1. Input: 48x48 grayscale image
2. Flatten: 2,304 input features
3. Dense layer: 128 neurons
4. ReLU activation
5. Output layer: 3 neurons
6. Softmax probabilities for classification

The model is trained using categorical cross-entropy loss with integer-encoded labels and mini-batch gradient descent.


### 1.3 Experimental Design

We treat Augmentation V1 and Augmentation V3 as separate
experiments using our NumPy neural-network baseline.

Each experiment uses its own predefined training,
validation, and test partitions.

To maintain consistency, both experiments use the same:

- Neural network architecture
- Weight initialization procedure
- Learning rate and optimizer
- Batch size
- Number of training epochs

The primary objective is to evaluate the baseline's
performance on each dataset configuration independently.

Because the validation and test partitions differ,
performance differences cannot be attributed exclusively
to the augmentation strategy.

### Data Leakage Prevention

Within each experiment, we must ensure that training,
validation, and test images remain independent.

We will first check for exact image-content overlap
between partitions. We will then investigate potential
near-duplicate or augmented-source leakage before training.


# -----------------------------------------------------------

In [3]:
# imports and environment verification

from pathlib import Path
import sys
import inspect
import numpy as np

import capstone_data.image_loader as loader

from capstone_nn.model import Sequential
from capstone_nn.layers import Flatten, Dense
from capstone_nn.activations import ReLU
from capstone_nn.training import train_epoch, evaluate

# Verify Python environment
print("Python executable:", sys.executable)

# Verify project directories
print("\nProject root:", loader.ROOT)
print("Data directory:", loader.DATA_DIR)

# Verify emotion classes
print("\nClasses:", loader.CLASS_NAMES)
print("Class labels:", loader.CLASS_TO_INDEX)

# Verify available datasets
print("\nAvailable datasets:")

for name, path in loader.list_datasets().items():
    print(f"  {name}: {path}")

# Verify that our updated loader supports split selection
print("\nLoader signature:")
print(inspect.signature(loader.load_dataset))


Python executable: C:\Users\Sheehizzle\miniconda3\envs\data4380\python.exe

Project root: C:\Users\Sheehizzle\Code\Capstone\Capstone-DATA4381
Data directory: C:\Users\Sheehizzle\Code\Capstone\Capstone-DATA4381\data

Classes: ('angry', 'happy', 'sad')
Class labels: {'angry': 0, 'happy': 1, 'sad': 2}

Available datasets:
  aug_data_v1: C:\Users\Sheehizzle\Code\Capstone\Capstone-DATA4381\data\aug_data_v1
  aug_data_v3: C:\Users\Sheehizzle\Code\Capstone\Capstone-DATA4381\data\aug_data_v3
  final: C:\Users\Sheehizzle\Code\Capstone\Capstone-DATA4381\data\final

Loader signature:
(datasets='final', split=None)



## 1. Dataset Loading and Integrity Verification

### 1.1 Loading the Experimental Datasets

This notebook evaluates our NumPy neural-network baseline using two independently prepared versions of the FER-2013 dataset: Augmentation V1 and Augmentation V3.

Each dataset contains three predefined partitions:

- **Training:** Used to optimize the neural network's parameters.
- **Validation:** Used to monitor generalization and guide model selection.
- **Testing:** Reserved for final evaluation.

We load each dataset using our `capstone_data` package without performing additional random splitting.

### Experimental Design

V1 and V3 are treated as **independent experiments**, each using its own training, validation, and test partitions.

Both experiments use the same neural-network architecture, initialization procedure, and training hyperparameters.

Because their evaluation partitions differ, differences in performance cannot be attributed exclusively to augmentation.

### Data Integrity

Before training, we will:

- Examine the class distributions of each dataset.
- Verify image dimensions and class labels.
- Check for exact duplicate images across partitions.
- Investigate potential near-duplicate and augmentation-related leakage.

These checks help establish whether each dataset is suitable for independent model evaluation.


In [7]:

# Define our experiments
versions = ("aug_data_v1", "aug_data_v3")
splits = ("train", "validation", "test")

# Load all six existing partitions
datasets = {}

for version in versions:
    datasets[version] = {}

    for split in splits:
        datasets[version][split] = loader.load_dataset(
            version,
            split=split
        )

        print(
            f"{version} / {split}: "
            f"{len(datasets[version][split]):,} images"
        )

print("\nLoaded dataset keys:", list(datasets.keys()))


aug_data_v1 / train: 14,019 images
aug_data_v1 / validation: 1,662 images
aug_data_v1 / test: 1,673 images
aug_data_v3 / train: 12,047 images
aug_data_v3 / validation: 1,650 images
aug_data_v3 / test: 1,646 images

Loaded dataset keys: ['aug_data_v1', 'aug_data_v3']


In [8]:

import hashlib
from PIL import Image

# Generate a fingerprint from an image's pixel contents
def fingerprint(record):

    with Image.open(record["path"]) as image:
        gray = image.convert("L")

        digest = hashlib.sha256()
        digest.update(str(gray.size).encode())
        digest.update(gray.tobytes())

    return digest.hexdigest()

# Check each experiment independently
for version in versions:

    print(f"\n{version.upper()} — LEAKAGE CHECK")
    print("=" * 50)

    # Store unique image hashes for each partition
    hashes = {}

    for split in splits:

        records = datasets[version][split]

        hashes[split] = {
            fingerprint(record)
            for record in records
        }

        print(
            f"{split:<12}: "
            f"{len(records):,} images, "
            f"{len(hashes[split]):,} unique hashes"
        )

    # Compare image contents across partitions
    overlaps = {
        "Train / Validation":
            hashes["train"] & hashes["validation"],

        "Train / Test":
            hashes["train"] & hashes["test"],

        "Validation / Test":
            hashes["validation"] & hashes["test"],
    }

    print("\nCross-partition overlap:")

    for comparison, shared in overlaps.items():
        print(f"{comparison:<22}: {len(shared)}")

    if all(len(shared) == 0 for shared in overlaps.values()):
        print("\nPASS: No exact pixel overlap detected.")
    else:
        print("\nWARNING: Exact pixel overlap detected.")



AUG_DATA_V1 — LEAKAGE CHECK
train       : 14,019 images, 13,721 unique hashes
validation  : 1,662 images, 1,644 unique hashes
test        : 1,673 images, 1,643 unique hashes

Cross-partition overlap:
Train / Validation    : 18
Train / Test          : 21
Validation / Test     : 1


AUG_DATA_V3 — LEAKAGE CHECK
train       : 12,047 images, 11,467 unique hashes
validation  : 1,650 images, 1,644 unique hashes
test        : 1,646 images, 1,643 unique hashes

Cross-partition overlap:
Train / Validation    : 12
Train / Test          : 17
Validation / Test     : 1



In [9]:

# Compare unique image contents across dataset versions.

for split in ("validation", "test"):

    v1_hashes = {
        fingerprint(record)
        for record in datasets["aug_data_v1"][split]
    }

    v3_hashes = {
        fingerprint(record)
        for record in datasets["aug_data_v3"][split]
    }

    print(f"\n{split.upper()}")
    print("=" * 40)

    print("V1 unique:", len(v1_hashes))
    print("V3 unique:", len(v3_hashes))

    print("Shared:", len(v1_hashes & v3_hashes))
    print("V1 only:", len(v1_hashes - v3_hashes))
    print("V3 only:", len(v3_hashes - v1_hashes))

    print("Same unique images:", v1_hashes == v3_hashes)



VALIDATION
V1 unique: 1644
V3 unique: 1644
Shared: 1644
V1 only: 0
V3 only: 0
Same unique images: True

TEST
V1 unique: 1643
V3 unique: 1643
Shared: 1643
V1 only: 0
V3 only: 0
Same unique images: True



### 1.5 Investigating Duplicate Images

Both augmentation versions contain identical sets of unique
validation and test image contents, despite having different
numbers of total images.

Our audit also identified exact image overlap between
training and evaluation partitions.

We will inspect the filenames and class labels of overlapping
images to investigate their origin.

No dataset files will be modified during this investigation.


In [10]:

from collections import defaultdict


def group_by_hash(records):
    """Group image records by their pixel fingerprint."""

    groups = defaultdict(list)

    for record in records:
        image_hash = fingerprint(record)
        groups[image_hash].append(record)

    return groups


for version in versions:

    print(f"\n{version.upper()}")
    print("=" * 55)

    # Build hash-to-record mappings for all partitions.
    groups = {
        split: group_by_hash(datasets[version][split])
        for split in splits
    }

    # Examine duplicates within evaluation partitions.
    for split in ("validation", "test"):

        duplicates = {
            h: records
            for h, records in groups[split].items()
            if len(records) > 1
        }

        extra_copies = sum(
            len(records) - 1
            for records in duplicates.values()
        )

        print(f"\n{split.upper()} DUPLICATES")
        print("Duplicate groups:", len(duplicates))
        print("Extra copies:", extra_copies)

    # Examine training/evaluation overlap.
    for split in ("validation", "test"):

        shared = (
            groups["train"].keys()
            & groups[split].keys()
        )

        print(f"\nTRAIN / {split.upper()} OVERLAP")
        print("Matching hashes:", len(shared))

        # Display three examples without overwhelming output.
        for image_hash in sorted(shared)[:3]:

            print("\n  Training files:")

            for record in groups["train"][image_hash][:3]:
                print(
                    "   ",
                    record["class_name"],
                    record["path"].name
                )

            print(f"  {split.capitalize()} files:")

            for record in groups[split][image_hash][:3]:
                print(
                    "   ",
                    record["class_name"],
                    record["path"].name
                )



AUG_DATA_V1

VALIDATION DUPLICATES
Duplicate groups: 18
Extra copies: 18

TEST DUPLICATES
Duplicate groups: 30
Extra copies: 30

TRAIN / VALIDATION OVERLAP
Matching hashes: 18

  Training files:
    angry original_001221.jpg
  Validation files:
    angry original_000391.jpg

  Training files:
    happy original_001602.jpg
  Validation files:
    happy original_000070.jpg

  Training files:
    happy original_004652.jpg
  Validation files:
    happy original_000923.jpg

TRAIN / TEST OVERLAP
Matching hashes: 21

  Training files:
    angry original_000503.jpg
  Test files:
    angry original_000355.jpg

  Training files:
    angry original_000101.jpg
  Test files:
    angry original_000433.jpg

  Training files:
    happy original_002419.jpg
  Test files:
    happy original_000228.jpg

AUG_DATA_V3

VALIDATION DUPLICATES
Duplicate groups: 6
Extra copies: 6

TEST DUPLICATES
Duplicate groups: 3
Extra copies: 3

TRAIN / VALIDATION OVERLAP
Matching hashes: 12

  Training files:
    angry ori

In [11]:

from collections import Counter

for version in versions:

    print(f"\n{version.upper()}")
    print("=" * 50)

    # Group images by their pixel fingerprints
    groups = {
        split: group_by_hash(datasets[version][split])
        for split in splits
    }

    for left, right in [
        ("train", "validation"),
        ("train", "test"),
        ("validation", "test"),
    ]:

        shared = groups[left].keys() & groups[right].keys()

        image_types = Counter()
        label_conflicts = 0

        for image_hash in shared:

            left_records = groups[left][image_hash]
            right_records = groups[right][image_hash]

            # Check filenames on both sides
            names = [
                record["path"].name.lower()
                for record in left_records + right_records
            ]

            if all(name.startswith("original_") for name in names):
                image_types["Original only"] += 1
            elif any(name.startswith("augmented_") for name in names):
                image_types["Includes augmented"] += 1
            else:
                image_types["Other filenames"] += 1

            # Check whether identical images have conflicting labels
            labels = {
                record["class_name"]
                for record in left_records + right_records
            }

            if len(labels) > 1:
                label_conflicts += 1

        print(f"\n{left} / {right}")
        print("Overlapping hashes:", len(shared))
        print("Image types:", dict(image_types))
        print("Conflicting labels:", label_conflicts)



AUG_DATA_V1

train / validation
Overlapping hashes: 18
Image types: {'Original only': 18}
Conflicting labels: 0

train / test
Overlapping hashes: 21
Image types: {'Original only': 21}
Conflicting labels: 0

validation / test
Overlapping hashes: 1
Image types: {'Original only': 1}
Conflicting labels: 0

AUG_DATA_V3

train / validation
Overlapping hashes: 12
Image types: {'Original only': 12}
Conflicting labels: 0

train / test
Overlapping hashes: 17
Image types: {'Original only': 17}
Conflicting labels: 0

validation / test
Overlapping hashes: 1
Image types: {'Original only': 1}
Conflicting labels: 0



### 1.6 Preparing Evaluation Partitions

The duplicate audit identified exact pixel overlap between
training, validation, and test partitions.

For exploratory experiments, we construct evaluation
partitions by:

1. Retaining one copy of each unique evaluation image.
2. Excluding images that appear in either training dataset.
3. Ensuring validation and test contain no shared images.
4. Preserving the original class labels.

The augmented training datasets remain unchanged.

These corrected evaluation partitions are used for
exploratory baseline assessment. A source-level,
duplicate-aware split is required for final reporting.


In [12]:

def unique_by_hash(records):
    """Retain one record per unique image content."""

    unique = {}

    for record in records:
        image_hash = fingerprint(record)

        if image_hash in unique:
            if unique[image_hash]["label"] != record["label"]:
                raise ValueError("Conflicting labels detected.")

        else:
            unique[image_hash] = record

    return unique


# Confirm V1 and V3 holdout images have identical labels.
for split in ("validation", "test"):

    v1 = unique_by_hash(datasets["aug_data_v1"][split])
    v3 = unique_by_hash(datasets["aug_data_v3"][split])

    v1_labels = {h: r["label"] for h, r in v1.items()}
    v3_labels = {h: r["label"] for h, r in v3.items()}

    if v1_labels != v3_labels:
        raise ValueError(
            f"V1 and V3 differ in {split} images or labels."
        )


# Collect every exact pixel hash used for training.
train_hashes = {
    fingerprint(record)
    for version in versions
    for record in datasets[version]["train"]
}

# Start with the shared unique evaluation images.
val_candidates = unique_by_hash(
    datasets["aug_data_v1"]["validation"]
)

test_candidates = unique_by_hash(
    datasets["aug_data_v1"]["test"]
)

# Exclude anything that appears in either training set.
clean_val = [
    record
    for h, record in val_candidates.items()
    if h not in train_hashes
]

clean_val_hashes = {
    fingerprint(record) for record in clean_val
}

# Also prevent validation/test overlap.
clean_test = [
    record
    for h, record in test_candidates.items()
    if h not in train_hashes
    and h not in clean_val_hashes
]

# Verify the resulting partitions.
assert all(
    fingerprint(record) not in train_hashes
    for record in clean_val + clean_test
)

assert not (
    clean_val_hashes
    & {fingerprint(record) for record in clean_test}
)

for name, records in [
    ("Validation", clean_val),
    ("Test", clean_test),
]:
    print(f"\n{name}: {len(records):,} images")

    counts = loader.count_images_by_class(records)

    for class_name in loader.CLASS_NAMES:
        print(f"  {class_name:<8}: {counts[class_name]:,}")

print("\nExact-overlap checks passed.")
print("Original dataset files were not modified.")



Validation: 1,626 images
  angry   : 454
  happy   : 992
  sad     : 180

Test: 1,621 images
  angry   : 451
  happy   : 989
  sad     : 181

Exact-overlap checks passed.
Original dataset files were not modified.



## 2. Data Preparation

### 2.1 Image Dimension Verification

Our neural-network baseline expects grayscale images with
dimensions of 48 × 48 pixels.

Before converting the datasets into NumPy arrays, we verify
that all images have the expected dimensions.

We examine:

- Augmentation V1 training images
- Augmentation V3 training images
- Cleaned validation images
- Cleaned test images

This check is particularly important because geometric
augmentation operations, such as rotation, can alter image
dimensions.

Any unexpected dimensions will be investigated before
training.

### 2.2 Normalization

After dimension verification, images will be converted to
NumPy arrays and normalized to the interval [0, 1] by
dividing pixel intensities by 255.

The model will receive arrays of shape:

- Images: (N, 48, 48)
- Labels: (N,)

Each image will subsequently be flattened into 2,304
features by our neural network's Flatten layer.


In [13]:

from collections import Counter
from PIL import Image

# Datasets we will use for training and evaluation
image_groups = {
    "V1 Training": datasets["aug_data_v1"]["train"],
    "V3 Training": datasets["aug_data_v3"]["train"],
    "Validation": clean_val,
    "Test": clean_test,
}

expected_size = (48, 48)

for name, records in image_groups.items():

    sizes = Counter()
    unexpected = []

    for record in records:

        with Image.open(record["path"]) as image:
            size = image.size

        sizes[size] += 1

        if size != expected_size and len(unexpected) < 3:
            unexpected.append(record["path"])

    print(f"\n{name}")
    print("=" * 40)

    for size, count in sorted(sizes.items()):
        print(f"{size}: {count:,} images")

    if not unexpected:
        print("PASS: All images are 48x48.")
    else:
        print("WARNING: Unexpected image dimensions.")

        for path in unexpected:
            print(" ", path)



V1 Training
(48, 48): 14,019 images
PASS: All images are 48x48.

V3 Training
(48, 48): 7,356 images
(50, 50): 924 images
(52, 52): 936 images
(54, 54): 989 images
(56, 56): 1,025 images
(58, 58): 817 images
  C:\Users\Sheehizzle\Code\Capstone\Capstone-DATA4381\data\aug_data_v3\angry\train\augmented_00000.jpg
  C:\Users\Sheehizzle\Code\Capstone\Capstone-DATA4381\data\aug_data_v3\angry\train\augmented_00001.jpg
  C:\Users\Sheehizzle\Code\Capstone\Capstone-DATA4381\data\aug_data_v3\angry\train\augmented_00003.jpg

Validation
(48, 48): 1,626 images
PASS: All images are 48x48.

Test
(48, 48): 1,621 images
PASS: All images are 48x48.



### 2.3 Handling Inconsistent Image Dimensions

Our dimension audit identified images larger than 48 × 48
pixels in the V3 training partition.

This is consistent with rotation augmentation using
an expanded image canvas.

To maintain compatibility with our neural network,
we added optional resizing to the `capstone_data`
image loader.

For V3, oversized images are resized to 48 × 48 using
bilinear interpolation during loading. V1 and the
evaluation partitions retain their original dimensions.

This preprocessing occurs in memory and does not
modify the underlying image files.

For final experiments, resizing should ideally be
standardized within the augmentation pipeline itself.


In [15]:

import importlib
from PIL import Image

# Reload our updated image loader
importlib.reload(loader)

# Find an oversized V3 training image
sample = None

for record in datasets["aug_data_v3"]["train"]:

    with Image.open(record["path"]) as image:
        if image.size != (48, 48):
            sample = record
            print("Original size:", image.size)
            break

if sample is None:
    raise ValueError("No oversized V3 image found.")

# Load and resize the image in memory
image_array = loader.load_image(
    sample["path"],
    resize=True,
)

print("Loaded shape:", image_array.shape)
print("Data type:", image_array.dtype)
print("Minimum value:", image_array.min())
print("Maximum value:", image_array.max())


Original size: (54, 54)
Loaded shape: (48, 48)
Data type: float32
Minimum value: 0.0
Maximum value: 0.99215686



### 2.4 Converting Images to NumPy Arrays

We convert both augmented training datasets and our
cleaned validation partition into NumPy arrays using
the `capstone_data.load_arrays()` function.

V1 images already satisfy the required 48 × 48 dimensions.

V3 images are resized to 48 × 48 when necessary using
bilinear interpolation.

Pixel intensities are normalized to the interval [0, 1]
and stored as 32-bit floating-point values.

Each dataset produces:

- **X:** Image array with shape (N, 48, 48)
- **y:** Integer class labels with shape (N,)

We reserve the test partition for final evaluation.

These experiments remain exploratory because exact-duplicate
cleanup does not eliminate all possible near-duplicate leakage.


In [17]:

# Load V1 training data
X_train_v1, y_train_v1 = loader.load_arrays(
    datasets["aug_data_v1"]["train"]
)

# Load V3 training data
# Oversized images are resized in memory.
X_train_v3, y_train_v3 = loader.load_arrays(
    datasets["aug_data_v3"]["train"],
    resize=True
)

# Load our cleaned validation partition
X_val, y_val = loader.load_arrays(clean_val)

# Verify array shapes and normalization
arrays = {
    "V1 Training": (X_train_v1, y_train_v1),
    "V3 Training": (X_train_v3, y_train_v3),
    "Validation": (X_val, y_val),
}

for name, (X, y) in arrays.items():

    print(f"\n{name}")
    print("=" * 40)

    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("Data type:", X.dtype)
    print("Pixel range:", X.min(), "to", X.max())

    assert X.shape == (len(y), 48, 48)
    assert X.dtype == np.float32
    assert np.isfinite(X).all()
    assert X.min() >= 0.0
    assert X.max() <= 1.0

print("\nAll NumPy arrays loaded successfully.")


Loading images nerds:   0%|          | 0/14019 [00:00<?, ?it/s]

Loading images nerds:   0%|          | 0/12047 [00:00<?, ?it/s]

Loading images nerds:   0%|          | 0/1626 [00:00<?, ?it/s]


V1 Training
X shape: (14019, 48, 48)
y shape: (14019,)
Data type: float32
Pixel range: 0.0 to 1.0

V3 Training
X shape: (12047, 48, 48)
y shape: (12047,)
Data type: float32
Pixel range: 0.0 to 1.0

Validation
X shape: (1626, 48, 48)
y shape: (1626,)
Data type: float32
Pixel range: 0.0 to 1.0

All NumPy arrays loaded successfully.



## 3. Neural Network Architecture

### 3.1 Baseline Model Initialization

We use our custom `capstone_nn` package to construct a fully
connected feedforward neural network from scratch using NumPy.

The architecture consists of:

1. **Flatten:** Converts each 48 × 48 image into 2,304 features.
2. **Dense(2304, 128):** Maps input features to 128 hidden neurons.
3. **ReLU:** Introduces nonlinearity.
4. **Dense(128, 3):** Produces three output scores (logits).

Softmax converts the logits into class probabilities during
training and evaluation.

We initialize model weights with a fixed random seed to
support reproducibility and consistent initialization
between V1 and V3 experiments.

Before training, we verify the model's output dimensions
and probability calculations.


In [18]:

from capstone_nn.losses import softmax

# Construct our baseline neural network.
def build_baseline(seed=42):

    # Dense layers use NumPy's global random generator.
    np.random.seed(seed)

    model = Sequential(
        Flatten(),
        Dense(2304, 128),
        ReLU(),
        Dense(128, 3),
    )

    return model


# Initialize our first model.
model_v1 = build_baseline(seed=42)

# Select eight training images for a forward-pass test.
X_sample = X_train_v1[:8]

# Compute raw output scores (logits).
logits = model_v1.forward(X_sample)

# Convert logits into probabilities.
probabilities = softmax(logits)

print("Input shape:", X_sample.shape)
print("Logits shape:", logits.shape)
print("Probabilities shape:", probabilities.shape)

print("\nFirst image probabilities:")
for class_name, probability in zip(
    loader.CLASS_NAMES,
    probabilities[0],
):
    print(f"{class_name:<8}: {probability:.4f}")

print("\nProbability sum:", probabilities[0].sum())

# Verify output properties.
assert logits.shape == (8, 3)
assert np.isfinite(probabilities).all()

np.testing.assert_allclose(
    probabilities.sum(axis=1),
    np.ones(8),
    atol=1e-6,
)

print("\nForward-pass checks passed.")


Input shape: (8, 48, 48)
Logits shape: (8, 3)
Probabilities shape: (8, 3)

First image probabilities:
angry   : 0.6416
happy   : 0.1017
sad     : 0.2567

Probability sum: 1.0

Forward-pass checks passed.



### 3.2 Model Parameters and Reproducibility

Our baseline contains two fully connected layers with
trainable weights and biases.

For a dense layer, the number of trainable parameters is:

\[
\text{Parameters} = (n_{\text{input}} \times n_{\text{output}}) + n_{\text{output}}
\]

The Flatten and ReLU layers contain no trainable parameters.

To ensure consistent experimental conditions, we initialize
separate models for V1 and V3 using the same random seed.

This ensures both experiments begin with identical weights
and biases, allowing us to examine how training on different
dataset configurations affects the resulting models.

The two models maintain independent parameters during training.


In [19]:

# Initialize V3 with the same seed as V1.
model_v3 = build_baseline(seed=42)

# Inspect trainable parameters.
total_parameters = 0

print("BASELINE MODEL PARAMETERS")
print("=" * 50)

for index, layer in enumerate(model_v1.layers):

    layer_name = type(layer).__name__

    if isinstance(layer, Dense):

        weights = layer.weights.size
        biases = layer.bias.size

        parameters = weights + biases
        total_parameters += parameters

        print(f"\nLayer {index}: {layer_name}")
        print("Weights shape:", layer.weights.shape)
        print("Bias shape:", layer.bias.shape)
        print("Parameters:", f"{parameters:,}")

    else:
        print(f"\nLayer {index}: {layer_name}")
        print("Parameters: 0")

print("\nTotal trainable parameters:",
      f"{total_parameters:,}")


# Verify identical initialization.
for layer_v1, layer_v3 in zip(
    model_v1.layers,
    model_v3.layers,
):

    if isinstance(layer_v1, Dense):

        assert np.array_equal(
            layer_v1.weights,
            layer_v3.weights,
        )

        assert np.array_equal(
            layer_v1.bias,
            layer_v3.bias,
        )

        # Ensure parameters are independent objects.
        assert not np.shares_memory(
            layer_v1.weights,
            layer_v3.weights,
        )

print("\nPASS: Both models have identical initial parameters.")
print("PASS: Model parameters are stored independently.")


BASELINE MODEL PARAMETERS

Layer 0: Flatten
Parameters: 0

Layer 1: Dense
Weights shape: (2304, 128)
Bias shape: (128,)
Parameters: 295,040

Layer 2: ReLU
Parameters: 0

Layer 3: Dense
Weights shape: (128, 3)
Bias shape: (3,)
Parameters: 387

Total trainable parameters: 295,427

PASS: Both models have identical initial parameters.
PASS: Model parameters are stored independently.
